# 第 3 周练习 —— 合成技术面试题数据集生成器

用 **Gradio** 搭一个小工具：按类别 / 难度 / 数量，调用 LLM 生成「技术面试问答」合成数据，并导出为 JSON。

## 练习目标（理念）

- **输入**：面试类别、难度、样本数、可选额外上下文、选用模型
- **输出**：带 `metadata` + `items` 的结构化 JSON（题、答、代码示例、追问等）
- **和本课第 3 周相关**：多模型路由（OpenAI / OpenRouter）、system/user prompt 设计、JSON 抽取与 Gradio Blocks UI

## 功能特性

- 交互界面选择类别、难度、生成条数
- 多类别：Python、数据结构、算法、机器学习、系统设计、LLM Engineering
- 由 LLM 生成贴近真实面试的问答
- 导出 JSON，便于当训练/评测/刷题材料

## 使用场景

- 面试陪练机器人的训练数据
- 技术测评的评测集
- 自学用的刷题材料生成

## 怎么跑

1. 准备 `.env`：设置 `OPENAI_API_KEY`（若以 `sk-or-` 开头则走 OpenRouter）
2. 从上到下运行单元格，最后 `app.launch()` 打开 Gradio
3. 在界面点 Generate，或先用 Examples 一键填参


In [ ]:
# ========== 导入：环境、正则、JSON、OpenAI、Gradio ==========

# 导入标准库 os：读环境变量 OPENAI_API_KEY
import os
# 导入标准库 re：用正则剥离模型输出里的 markdown 代码围栏、修尾逗号
import re
# 导入标准库 json：解析/序列化面试题 JSON
import json
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端（可配 base_url 走 OpenRouter）
from openai import OpenAI
# 导入 Gradio：用 Blocks 搭数据集生成 UI
import gradio as gr


In [ ]:
# ========== 环境与客户端：OpenAI 直连 或 OpenRouter ==========

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 API Key（本练习变量名是 OPENAI_API_KEY）
openai_api_key = os.getenv('OPENAI_API_KEY')

# 有 key 则打印前 8 字符做存在性确认（勿泄露完整密钥）
if openai_api_key:
    print(f"API Key exists and begins {openai_api_key[:8]}")
else:
    print("API Key not set")

# 按 key 前缀判断走 OpenRouter 还是官方 OpenAI
if openai_api_key and openai_api_key.startswith('sk-or-'):
    # OpenRouter：同一个 OpenAI SDK，换 base_url 即可路由多家模型
    client = OpenAI(
        api_key=openai_api_key,
        base_url="https://openrouter.ai/api/v1"
    )
    print("Using OpenRouter")
else:
    # 直连 OpenAI：默认从环境变量读 key，不显式传 base_url
    client = OpenAI()
    print("Using OpenAI directly")

# 界面显示名 → 实际 model id（字符串影响计费与路由，保持原样）
MODELS = {
    "GPT-4.1-mini": "gpt-4.1-mini",
    "GPT-4.1": "gpt-4.1",
    "Claude Sonnet": "anthropic/claude-sonnet-4",
}
# 下拉框默认模型显示名
DEFAULT_MODEL = "GPT-4.1-mini"


In [ ]:
# ========== 面试类别字典：描述 / JSON schema 提示 / 话题列表 ==========

# CATEGORIES：每个类别给模型「出什么题、字段长什么样、可覆盖哪些话题」
# description / schema / topics 会进 system prompt，字符串保持英文原文
CATEGORIES = {
    "Python Fundamentals": {
        "description": "Core Python language features, syntax, and best practices",
        "schema": "Each object: question, answer (detailed explanation), code_example (if applicable), follow_up_questions (array of 2 strings), difficulty, tags (array)",
        "topics": ["decorators", "generators", "context managers", "GIL", "memory management", "metaclasses", "descriptors", "type hints"]
    },
    "Data Structures": {
        "description": "Data structures, implementations, and complexity analysis",
        "schema": "Each object: question, answer, code_example, time_complexity, space_complexity, follow_up_questions, difficulty, tags",
        "topics": ["hash tables", "binary trees", "heaps", "graphs", "linked lists", "tries", "stacks", "queues"]
    },
    "Algorithms": {
        "description": "Algorithmic problem-solving, optimization, and complexity",
        "schema": "Each object: question, answer, code_example, time_complexity, approach (e.g. divide-and-conquer, DP), follow_up_questions, difficulty, tags",
        "topics": ["sorting", "searching", "dynamic programming", "greedy", "graph algorithms", "recursion", "backtracking"]
    },
    "Machine Learning": {
        "description": "ML concepts, models, evaluation, and practical implementation",
        "schema": "Each object: question, answer, code_example (sklearn/pytorch if applicable), key_concepts (array), follow_up_questions, difficulty, tags",
        "topics": ["supervised learning", "unsupervised learning", "neural networks", "regularization", "cross-validation", "feature engineering", "model evaluation"]
    },
    "System Design": {
        "description": "System architecture, scalability, and distributed systems",
        "schema": "Each object: question, answer, key_components (array), trade_offs, follow_up_questions, difficulty, tags",
        "topics": ["load balancing", "caching", "databases", "message queues", "microservices", "API design", "scalability"]
    },
    "LLM Engineering": {
        "description": "LLM APIs, prompt engineering, RAG, and AI application development",
        "schema": "Each object: question, answer, code_example (API usage if applicable), best_practices (array), follow_up_questions, difficulty, tags",
        "topics": ["prompt engineering", "RAG", "fine-tuning", "embeddings", "tokenization", "agents", "function calling", "context management"]
    }
}

# 难度选项：会原样传给 get_system_prompt / 元数据
DIFFICULTY_LEVELS = ["Easy", "Medium", "Hard", "Mixed"]

# 打印已注册类别名，确认字典加载成功
print(f"Categories: {list(CATEGORIES.keys())}")


In [ ]:
# ========== 拼装 system prompt：按类别 / 难度 / 条数生成出题指令 ==========

def get_system_prompt(category: str, difficulty: str, num_samples: int) -> str:
    """按类别与难度拼出给 LLM 的 system prompt（返回的英文指令勿改译）。"""
    # 取出该类别的描述、schema、话题；未知类别则用空 dict，后面走默认 schema
    cat_info = CATEGORIES.get(category, {})
    schema = cat_info.get("schema", "Each object: question, answer, code_example, follow_up_questions, difficulty, tags")
    topics = cat_info.get("topics", [])
    
    # 按难度写一段英文 guidance，嵌入最终 prompt
    difficulty_guidance = ""
    if difficulty == "Easy":
        difficulty_guidance = "Questions should be suitable for junior developers, focusing on fundamentals."
    elif difficulty == "Medium":
        difficulty_guidance = "Questions should be suitable for mid-level developers, requiring practical experience."
    elif difficulty == "Hard":
        difficulty_guidance = "Questions should be suitable for senior developers, requiring deep expertise."
    else:  # Mixed
        difficulty_guidance = "Include a mix of easy, medium, and hard questions."
    
    # f-string：把条数、类别、schema 等插进英文模板；双大括号转义为字面 {
    return f"""You are a technical interview expert. Generate exactly {num_samples} realistic technical interview questions and comprehensive answers.

Category: {category}
Description: {cat_info.get('description', category)}
Relevant topics: {', '.join(topics)}

Difficulty: {difficulty}
{difficulty_guidance}

Schema (use these field names): {schema}

Requirements:
- Questions should be realistic interview questions asked at top tech companies
- Answers should be thorough, accurate, and demonstrate expertise
- Include practical code examples where applicable
- Follow-up questions should probe deeper understanding

Output valid JSON only: a single object with key "items" containing an array of {num_samples} objects.
No markdown code fences, no explanations. Just the JSON object.

Example structure:
{{
  "items": [
    {{
      "question": "...",
      "answer": "...",
      "code_example": "...",
      "follow_up_questions": ["...", "..."],
      "difficulty": "medium",
      "tags": ["...", "..."]
    }}
  ]
}}"""


In [ ]:
# ========== JSON 抽取助手：从模型原文里抠出题目数组 ==========

def extract_json_array(text: str) -> list:
    """从模型输出中提取 JSON 数组（兼容外层对象包一层 items 等字段）。"""
    # 空输出直接报错，避免后面静默失败
    if not text or not text.strip():
        raise ValueError("Empty model output.")
    
    # 去掉首尾空白，便于匹配围栏与花括号
    t = text.strip()
    
    # 若模型包了 ```json ... ```，用正则剥掉代码围栏
    if "```" in t:
        t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE | re.MULTILINE)
        t = re.sub(r"\s*```$", "", t, flags=re.MULTILINE)
        t = t.strip()
    
    # 先尝试整段 json.loads
    try:
        obj = json.loads(t)
    except json.JSONDecodeError:
        # 失败则截取第一个 { 到最后一个 }，并去掉非法尾逗号后再解析
        start = t.find("{")
        end = t.rfind("}")
        if start != -1 and end != -1 and end > start:
            t = t[start:end+1]
            # 修 JSON 里 `,]` / `,}` 这类尾逗号
            t = re.sub(r",\s*([\]}])", r"\1", t)
            obj = json.loads(t)
        else:
            raise ValueError("No valid JSON found in output.")
    
    # 已是 list：直接当作题目数组
    if isinstance(obj, list):
        return obj
    
    # 常见包装字段名：命中则取出内部 list
    for key in ("items", "data", "entries", "results", "questions"):
        if isinstance(obj.get(key), list):
            return obj[key]
    
    # 都不像数组结构则报错
    raise ValueError("JSON object has no recognizable array field.")


In [ ]:
# ========== 主生成函数：调 Chat Completions，返回格式化 JSON 字符串 ==========

def generate_interview_data(
    category: str,
    difficulty: str,
    num_samples: int,
    additional_context: str,
    model_name: str
) -> str:
    """生成合成技术面试 Q&A；成功/失败都返回 JSON 字符串给 Gradio 展示。"""
    
    # 未选类别：返回错误 JSON（文案保持英文，供界面直接显示）
    if not category:
        return json.dumps({"error": "Please select a category."}, indent=2)
    
    # 把条数夹在 1～15，防止一次请求过大
    num_samples = max(1, min(15, int(num_samples)))
    
    # 按当前参数拼 system prompt
    system_prompt = get_system_prompt(category, difficulty, num_samples)
    
    # 用户有额外说明就用；否则给一句默认英文 user prompt
    user_prompt = additional_context.strip() if additional_context else f"Generate {num_samples} high-quality {category} interview questions."
    
    # 显示名 → model id；未知则回退默认模型
    model_id = MODELS.get(model_name, MODELS[DEFAULT_MODEL])
    
    try:
        # 控制台进度提示（不影响返回给 UI 的 JSON）
        print(f"Generating {num_samples} {difficulty} {category} questions using {model_name}...")
        
        # 非流式一次拿完整回复；temperature/max_tokens 保持原参数
        response = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.7,
            max_tokens=4000
        )
        
        # 取出助手文本；None 时当成空串
        raw_content = response.choices[0].message.content or ""
        
        # 从原文抽取题目 list
        items = extract_json_array(raw_content)
        
        # 包一层 metadata，便于溯源（类别/难度/条数/模型）
        result = {
            "metadata": {
                "category": category,
                "difficulty": difficulty,
                "count": len(items),
                "model": model_name
            },
            "items": items
        }
        
        print(f"Successfully generated {len(items)} items.")
        # indent=2：给 Gradio Code 组件好看展示
        return json.dumps(result, indent=2)
        
    except Exception as e:
        # 任意失败都变成带 hint 的 JSON，避免 UI 直接炸栈
        error_result = {
            "error": str(e),
            "hint": "Check your API key and try again."
        }
        return json.dumps(error_result, indent=2)


In [ ]:
# ========== Gradio Blocks：类别 / 难度 / 条数 / 模型 / 生成按钮 ==========

# Soft 主题的 Blocks 应用；title 会出现在浏览器标签
with gr.Blocks(title="Technical Interview Data Generator", theme=gr.themes.Soft()) as app:
    # 页头说明（界面文案保留英文；教学说明已在上方 markdown 格）
    gr.Markdown("""
    # Technical Interview Dataset Generator
    
    Generate synthetic technical interview Q&A data for training, evaluation, or study purposes.
    
    **How to use:**
    1. Select a category (Python, Data Structures, ML, etc.)
    2. Choose difficulty level
    3. Set number of samples (1-15)
    4. Optionally add specific topics or context
    5. Click Generate
    """)
    
    # 两列布局：左类别+难度，右条数+模型
    with gr.Row():
        with gr.Column():
            # 类别下拉：选项来自 CATEGORIES 的 key
            category_dropdown = gr.Dropdown(
                choices=list(CATEGORIES.keys()),
                value=list(CATEGORIES.keys())[0],
                label="Category"
            )
            
            # 难度下拉：默认 Medium
            difficulty_dropdown = gr.Dropdown(
                choices=DIFFICULTY_LEVELS,
                value="Medium",
                label="Difficulty"
            )
        
        with gr.Column():
            # 生成条数滑块：1～15，步进 1
            num_samples_slider = gr.Slider(
                minimum=1,
                maximum=15,
                value=5,
                step=1,
                label="Number of Questions"
            )
            
            # 模型下拉：显示名来自 MODELS
            model_dropdown = gr.Dropdown(
                choices=list(MODELS.keys()),
                value=DEFAULT_MODEL,
                label="Model"
            )
    
    # 可选额外上下文：会作为 user_prompt（或覆盖默认 user 句）
    context_input = gr.Textbox(
        label="Additional Context (optional)",
        placeholder="e.g., Focus on async/await patterns, or Include questions about PyTorch",
        lines=2
    )
    
    # 主按钮：触发 generate_interview_data
    generate_btn = gr.Button("Generate Dataset", variant="primary")
    
    # 用 Code 组件展示 JSON 结果（带语法高亮）
    output_json = gr.Code(
        label="Generated Dataset (JSON)",
        language="json",
        lines=25
    )
    
    # 示例行：一键填入 inputs，方便演示
    gr.Markdown("### Examples")
    gr.Examples(
        examples=[
            ["Python Fundamentals", "Medium", 5, "Focus on decorators and context managers", "GPT-4.1-mini"],
            ["Machine Learning", "Hard", 3, "Deep learning and neural network architecture questions", "GPT-4.1-mini"],
            ["System Design", "Hard", 3, "Distributed systems and scalability", "GPT-4.1-mini"],
            ["LLM Engineering", "Medium", 5, "RAG and prompt engineering best practices", "GPT-4.1-mini"],
        ],
        inputs=[category_dropdown, difficulty_dropdown, num_samples_slider, context_input, model_dropdown]
    )
    
    # 把按钮点击接到生成函数：五个输入 → JSON 输出
    generate_btn.click(
        fn=generate_interview_data,
        inputs=[category_dropdown, difficulty_dropdown, num_samples_slider, context_input, model_dropdown],
        outputs=output_json
    )


In [ ]:
# ========== 启动 Gradio 应用 ==========

# launch()：起本地 Web 服务；在浏览器打开生成器界面
app.launch()


## 用法说明

### 输出格式

生成的 JSON 包含：

- **metadata**：类别、难度、条数、所用模型
- **items**：问答对象数组，字段通常包括：
  - `question`：面试题
  - `answer`：较完整的解答
  - `code_example`：适用时的示例代码
  - `follow_up_questions`：追问列表
  - `difficulty`：easy / medium / hard
  - `tags`：相关话题标签

（不同类别的 schema 可能多出 `time_complexity`、`key_components` 等字段，见 `CATEGORIES`。）

### 如何保存数据

把界面里的 JSON 复制存盘，或在代码里：

```python
import json
data = json.loads(output_string)
with open('interview_data.json', 'w') as f:
    json.dump(data, f, indent=2)
```

### 可用类别

- **Python Fundamentals**：装饰器、生成器、GIL、内存管理
- **Data Structures**：树、图、哈希表、复杂度分析
- **Algorithms**：排序、动态规划、图算法、优化
- **Machine Learning**：模型、评估、特征工程
- **System Design**：可扩展性、分布式、架构
- **LLM Engineering**：Prompt、RAG、Embeddings、Agents
